# CNN Model Implementation for Alert Triage

This notebook implements a 1D Convolutional Neural Network (CNN) for binary alert classification.
The CNN extends the XGBoost baseline by learning hierarchical feature representations automatically.

**Key Objectives:**
1. Implement 1D CNN architecture for tabular network flow data
2. Train model with proper validation split
3. Evaluate performance and compare with XGBoost baseline
4. Analyze learned representations and feature importance

## Why CNN for Network Intrusion Detection?

**Advantages over XGBoost:**
- Automatic feature learning (no manual feature engineering)
- Hierarchical pattern recognition (low-level → high-level features)
- Translation invariance (similar patterns detected regardless of position)
- End-to-end learning (optimize entire pipeline jointly)

**1D CNN for Tabular Data:**
- Treats 194 features as a 1D sequence
- Convolutional filters learn local patterns across features
- Pooling layers reduce dimensionality
- Dense layers make final classification decision

## 1. Setup and Data Loading

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns

# Deep Learning
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, callbacks
from tensorflow.keras.utils import plot_model

# Scikit-learn for metrics and preprocessing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_auc_score, roc_curve
)

# Set random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU available: {tf.config.list_physical_devices('GPU')}")

In [ ]:
# Load preprocessed data
PROCESSED_DIR = Path("../data/processed")

X_train = pd.read_csv(PROCESSED_DIR / "X_train.csv")
X_test = pd.read_csv(PROCESSED_DIR / "X_test.csv")
y_train = pd.read_csv(PROCESSED_DIR / "y_train.csv").squeeze()
y_test = pd.read_csv(PROCESSED_DIR / "y_test.csv").squeeze()

print(f"Training set: {X_train.shape}")
print(f"Testing set: {X_test.shape}")
print(f"\nClass distribution (training):")
print(y_train.value_counts(normalize=True))

## 2. Data Preprocessing for CNN

**Key Steps:**
1. **Feature Scaling:** CNNs benefit from normalized inputs (unlike XGBoost)
2. **Validation Split:** Create validation set for model selection and early stopping
3. **Reshape for Conv1D:** Add channel dimension (batch_size, sequence_length, channels)

In [ ]:
# Feature scaling (fit only on training data)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Feature scaling complete.")
print(f"Mean of first feature (before): {X_train.iloc[:, 0].mean():.4f}")
print(f"Mean of first feature (after): {X_train_scaled[:, 0].mean():.4f}")
print(f"Std of first feature (after): {X_train_scaled[:, 0].std():.4f}")

In [ ]:
# Create validation split (20% of training data)
X_train_final, X_val, y_train_final, y_val = train_test_split(
    X_train_scaled, y_train,
    test_size=0.2,
    stratify=y_train,  # Maintain class distribution
    random_state=42
)

print(f"Training set: {X_train_final.shape}")
print(f"Validation set: {X_val.shape}")
print(f"Test set: {X_test_scaled.shape}")
print(f"\nValidation class distribution:")
print(pd.Series(y_val).value_counts(normalize=True))

In [ ]:
# Reshape for Conv1D: (samples, sequence_length, channels)
# We treat 194 features as a sequence with 1 channel
X_train_cnn = X_train_final.reshape(-1, 194, 1)
X_val_cnn = X_val.reshape(-1, 194, 1)
X_test_cnn = X_test_scaled.reshape(-1, 194, 1)

# Convert labels to numpy arrays
y_train_final = np.array(y_train_final)
y_val = np.array(y_val)
y_test_final = np.array(y_test)

print(f"CNN input shape: {X_train_cnn.shape}")
print(f"Interpretation: {X_train_cnn.shape[0]} samples, {X_train_cnn.shape[1]} features, {X_train_cnn.shape[2]} channel(s)")

## 3. CNN Architecture Design

**Architecture Philosophy:**
- **Conv Blocks:** Extract hierarchical features with increasing abstraction
- **Batch Normalization:** Stabilize training and improve convergence
- **Pooling:** Reduce dimensionality and provide translation invariance
- **Dropout:** Prevent overfitting through random deactivation
- **Dense Layers:** Learn non-linear combinations for classification

**Design Choices:**
- 3 convolutional blocks (64 → 128 → 256 filters)
- Kernel size = 3 (learn patterns across 3 adjacent features)
- ReLU activation (standard for CNNs)
- Global Max Pooling (select most important features)
- Binary classification output (sigmoid activation)

In [ ]:
def create_cnn_model(input_shape=(194, 1), name="CNN_AlertTriage"):
    """
    Create 1D CNN for binary alert classification.
    
    Architecture:
    - 3 Conv1D blocks with increasing filters (64→128→256)
    - Batch normalization and dropout for regularization
    - Global max pooling for dimensionality reduction
    - Dense layers for classification
    
    Args:
        input_shape: Tuple of (sequence_length, channels)
        name: Model name for identification
    
    Returns:
        Compiled Keras model
    """
    model = models.Sequential(name=name)
    
    # Input layer
    model.add(layers.Input(shape=input_shape, name='input'))
    
    # Conv Block 1: Learn low-level patterns
    model.add(layers.Conv1D(
        filters=64,
        kernel_size=3,
        activation='relu',
        padding='same',
        name='conv1'
    ))
    model.add(layers.BatchNormalization(name='bn1'))
    model.add(layers.MaxPooling1D(pool_size=2, name='pool1'))
    model.add(layers.Dropout(0.3, name='dropout1'))
    
    # Conv Block 2: Learn mid-level patterns
    model.add(layers.Conv1D(
        filters=128,
        kernel_size=3,
        activation='relu',
        padding='same',
        name='conv2'
    ))
    model.add(layers.BatchNormalization(name='bn2'))
    model.add(layers.MaxPooling1D(pool_size=2, name='pool2'))
    model.add(layers.Dropout(0.3, name='dropout2'))
    
    # Conv Block 3: Learn high-level patterns
    model.add(layers.Conv1D(
        filters=256,
        kernel_size=3,
        activation='relu',
        padding='same',
        name='conv3'
    ))
    model.add(layers.BatchNormalization(name='bn3'))
    model.add(layers.GlobalMaxPooling1D(name='global_pool'))
    
    # Dense layers for classification
    model.add(layers.Dense(128, activation='relu', name='dense1'))
    model.add(layers.Dropout(0.5, name='dropout3'))
    model.add(layers.Dense(64, activation='relu', name='dense2'))
    model.add(layers.Dropout(0.3, name='dropout4'))
    
    # Output layer
    model.add(layers.Dense(1, activation='sigmoid', name='output'))
    
    return model

In [ ]:
# Create model instance
cnn_model = create_cnn_model()

# Display architecture
cnn_model.summary()

In [ ]:
# Visualize architecture (optional)
try:
    plot_model(
        cnn_model,
        to_file='../models/cnn_architecture.png',
        show_shapes=True,
        show_layer_names=True,
        rankdir='TB',
        dpi=96
    )
    print("Architecture diagram saved to: ../models/cnn_architecture.png")
except:
    print("Could not generate architecture diagram (graphviz required)")

## 4. Model Compilation

**Compilation Parameters:**
- **Optimizer:** Adam (adaptive learning rate, works well for CNNs)
- **Loss:** Binary crossentropy (standard for binary classification)
- **Metrics:** Accuracy, Precision, Recall, AUC (same as XGBoost for comparison)

In [ ]:
# Compile model
cnn_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=[
        'accuracy',
        keras.metrics.Precision(name='precision'),
        keras.metrics.Recall(name='recall'),
        keras.metrics.AUC(name='auc')
    ]
)

print("Model compiled successfully!")

## 5. Training Callbacks

**Callbacks for Better Training:**
- **EarlyStopping:** Stop training when validation loss stops improving
- **ModelCheckpoint:** Save best model based on validation AUC
- **ReduceLROnPlateau:** Reduce learning rate when validation loss plateaus
- **TensorBoard:** Log training metrics for visualization (optional)

In [ ]:
# Create models directory
MODEL_DIR = Path("../models")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

# Define callbacks
early_stopping = callbacks.EarlyStopping(
    monitor='val_auc',
    patience=10,
    mode='max',
    restore_best_weights=True,
    verbose=1
)

model_checkpoint = callbacks.ModelCheckpoint(
    filepath=str(MODEL_DIR / 'cnn_best_model.keras'),
    monitor='val_auc',
    mode='max',
    save_best_only=True,
    verbose=1
)

reduce_lr = callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=5,
    min_lr=1e-7,
    verbose=1
)

callback_list = [early_stopping, model_checkpoint, reduce_lr]

print("Callbacks configured:")
print("- Early stopping on validation AUC (patience=10)")
print("- Model checkpoint saving best model")
print("- Learning rate reduction on plateau")

## 6. Model Training

**Training Configuration:**
- Epochs: 100 (will stop early if validation doesn't improve)
- Batch size: 128 (balance between speed and stability)
- Validation: Monitor on 20% validation split
- Class weights: Handle class imbalance (68% attacks, 32% benign)

In [ ]:
# Calculate class weights to handle imbalance
from sklearn.utils import class_weight

class_weights = class_weight.compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train_final),
    y=y_train_final
)

class_weight_dict = {i: weight for i, weight in enumerate(class_weights)}

print(f"Class weights: {class_weight_dict}")
print(f"This gives more weight to benign class (minority) during training")

In [ ]:
# Train model
print("Starting training...\n")

history = cnn_model.fit(
    X_train_cnn, y_train_final,
    validation_data=(X_val_cnn, y_val),
    epochs=100,
    batch_size=128,
    class_weight=class_weight_dict,
    callbacks=callback_list,
    verbose=1
)

print("\nTraining complete!")

## 7. Training History Visualization

In [ ]:
# Plot training history
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Loss
axes[0, 0].plot(history.history['loss'], label='Training Loss')
axes[0, 0].plot(history.history['val_loss'], label='Validation Loss')
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('Loss')
axes[0, 0].set_title('Model Loss')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Accuracy
axes[0, 1].plot(history.history['accuracy'], label='Training Accuracy')
axes[0, 1].plot(history.history['val_accuracy'], label='Validation Accuracy')
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].set_ylabel('Accuracy')
axes[0, 1].set_title('Model Accuracy')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Precision
axes[1, 0].plot(history.history['precision'], label='Training Precision')
axes[1, 0].plot(history.history['val_precision'], label='Validation Precision')
axes[1, 0].set_xlabel('Epoch')
axes[1, 0].set_ylabel('Precision')
axes[1, 0].set_title('Model Precision')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# Recall
axes[1, 1].plot(history.history['recall'], label='Training Recall')
axes[1, 1].plot(history.history['val_recall'], label='Validation Recall')
axes[1, 1].set_xlabel('Epoch')
axes[1, 1].set_ylabel('Recall')
axes[1, 1].set_title('Model Recall')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../models/cnn_training_history.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Best validation AUC: {max(history.history['val_auc']):.4f}")
print(f"Training stopped at epoch: {len(history.history['loss'])}")

## 8. Model Evaluation on Test Set

In [ ]:
# Load best model
best_model = keras.models.load_model(MODEL_DIR / 'cnn_best_model.keras')

# Generate predictions
y_pred_proba = best_model.predict(X_test_cnn, verbose=0).flatten()
y_pred = (y_pred_proba >= 0.5).astype(int)

print("Predictions generated on test set.")
print(f"Prediction shape: {y_pred.shape}")
print(f"Probability range: [{y_pred_proba.min():.4f}, {y_pred_proba.max():.4f}]")

In [ ]:
# Calculate metrics
accuracy = accuracy_score(y_test_final, y_pred)
precision = precision_score(y_test_final, y_pred)
recall = recall_score(y_test_final, y_pred)
f1 = f1_score(y_test_final, y_pred)
auc = roc_auc_score(y_test_final, y_pred_proba)

print("\n" + "="*60)
print("CNN MODEL PERFORMANCE ON TEST SET")
print("="*60)
print(f"Accuracy:  {accuracy:.4f} ({accuracy*100:.2f}%)")
print(f"Precision: {precision:.4f} ({precision*100:.2f}%)")
print(f"Recall:    {recall:.4f} ({recall*100:.2f}%)")
print(f"F1-Score:  {f1:.4f} ({f1*100:.2f}%)")
print(f"ROC-AUC:   {auc:.4f}")
print("="*60 + "\n")

In [ ]:
# Compare with XGBoost baseline
print("\n" + "="*60)
print("COMPARISON: CNN vs XGBoost Baseline")
print("="*60)

xgb_results = {
    'Accuracy': 0.8756,
    'Precision': 0.8233,
    'Recall': 0.9855,
    'F1-Score': 0.8971,
    'ROC-AUC': 0.9332
}

cnn_results = {
    'Accuracy': accuracy,
    'Precision': precision,
    'Recall': recall,
    'F1-Score': f1,
    'ROC-AUC': auc
}

comparison_df = pd.DataFrame({
    'XGBoost': xgb_results,
    'CNN': cnn_results,
    'Difference': {k: cnn_results[k] - xgb_results[k] for k in xgb_results}
})

print(comparison_df.to_string())
print("\nNote: Positive difference means CNN performs better")
print("="*60 + "\n")

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test_final, y_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Benign', 'Attack'],
            yticklabels=['Benign', 'Attack'])
plt.title('CNN Confusion Matrix')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.savefig('../models/cnn_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

# Print detailed breakdown
tn, fp, fn, tp = cm.ravel()
print("\nConfusion Matrix Breakdown:")
print(f"True Negatives (TN):  {tn:,} - Correctly identified benign")
print(f"False Positives (FP): {fp:,} - Benign flagged as attack")
print(f"False Negatives (FN): {fn:,} - Attacks missed")
print(f"True Positives (TP):  {tp:,} - Correctly identified attacks")
print(f"\nFalse Negative Rate: {fn/(fn+tp)*100:.2f}% (attacks missed)")
print(f"False Positive Rate: {fp/(fp+tn)*100:.2f}% (false alarms)")

In [ ]:
# Classification report
print("\nClassification Report:")
print(classification_report(y_test_final, y_pred, 
                          target_names=['Benign', 'Attack'],
                          digits=4))

In [ ]:
# ROC Curve
fpr, tpr, thresholds = roc_curve(y_test_final, y_pred_proba)

plt.figure(figsize=(10, 6))
plt.plot(fpr, tpr, label=f'CNN (AUC = {auc:.4f})', linewidth=2)
plt.plot([0, 1], [0, 1], 'k--', label='Random Classifier', linewidth=1)
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate (Recall)')
plt.title('ROC Curve - CNN Model')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../models/cnn_roc_curve.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Save Results and Model Artifacts

In [ ]:
# Save results
results = {
    'model': 'CNN',
    'accuracy': float(accuracy),
    'precision': float(precision),
    'recall': float(recall),
    'f1_score': float(f1),
    'roc_auc': float(auc),
    'true_positives': int(tp),
    'false_positives': int(fp),
    'true_negatives': int(tn),
    'false_negatives': int(fn),
    'training_samples': int(X_train_final.shape[0]),
    'validation_samples': int(X_val.shape[0]),
    'test_samples': int(X_test_scaled.shape[0]),
    'num_features': 194,
    'epochs_trained': len(history.history['loss']),
    'best_epoch': int(np.argmax(history.history['val_auc'])) + 1
}

# Save as JSON
import json
with open(MODEL_DIR / 'cnn_results.json', 'w') as f:
    json.dump(results, f, indent=2)

print("Results saved to: ../models/cnn_results.json")

# Save predictions for further analysis
predictions_df = pd.DataFrame({
    'y_true': y_test_final,
    'y_pred': y_pred,
    'y_pred_proba': y_pred_proba
})
predictions_df.to_csv(MODEL_DIR / 'cnn_predictions.csv', index=False)

print("Predictions saved to: ../models/cnn_predictions.csv")
print("\nAll artifacts saved successfully!")

## 10. Summary and Next Steps

**Model Performance:**
- The CNN model has been trained and evaluated
- Results compared with XGBoost baseline
- All metrics and visualizations saved

**Key Observations:**
1. Check if CNN outperforms or underperforms XGBoost
2. Analyze which types of attacks CNN catches better/worse
3. Evaluate computational cost (training time, model size)

**Next Steps:**
1. ✅ CNN implementation complete
2. 🔄 Implement LSTM for temporal sequence analysis
3. 📊 Comprehensive comparison of all three models
4. 💻 Dashboard development with model integration

**Files Generated:**
- `../models/cnn_best_model.keras` - Best trained model
- `../models/cnn_results.json` - Performance metrics
- `../models/cnn_predictions.csv` - Test set predictions
- `../models/cnn_training_history.png` - Training curves
- `../models/cnn_confusion_matrix.png` - Confusion matrix
- `../models/cnn_roc_curve.png` - ROC curve